# Vanilla Term Extraction Baseline

Simple word-boundary matching baseline:
- Extract all unique terms from training data (lowercased)
- Match terms in test sentences using word boundaries
- No machine learning, just exact substring matching

In [3]:
import pandas as pd
import json
import os

data_path = "../data"
train_data_path = os.path.join(data_path, "taskA/train.json")

with open(train_data_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

terms = set()
for id, content in train_data.items():
    for term in content['terms']:
        terms.add(term.lower())

terms = sorted(terms)
print(len(terms))

5525


## Load Test Data and Predict Terms

In [3]:
import re
from tqdm import tqdm

test_data_path = os.path.join(data_path, "taskA/test.json")

with open(test_data_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

dump_pred = {}

for id, content in tqdm(test_data.items(), desc="Processing test data", total=len(test_data)):
    dump_pred[id] = {
        "text": content['text'],
        "terms": [],
    }
    text = content['text'].lower()
    for term in terms:
        # Use word boundary matching to find complete words only
        pattern = r'\b' + re.escape(term) + r'\b'
        if re.search(pattern, text):
            dump_pred[id]['terms'].append(term)

with open("predictions/subtaskA_a_vanilla_preds.json", "w", encoding="utf-8") as f:
    json.dump(dump_pred, f, ensure_ascii=False, indent=4)

Processing test data: 100%|██████████| 567/567 [02:35<00:00,  3.64it/s]


## Evaluation Functions

Using the official competition evaluation metrics.

In [4]:
def micro_f1_score(gold_standard, system_output):
    """
    Evaluates performance using Precision, Recall, and F1 score 
    based on individual term matching (micro-average).
    """
    total_true_positives = 0
    total_false_positives = 0
    total_false_negatives = 0
    
    for gold, system in zip(gold_standard, system_output):
        gold_set = set(gold)
        system_set = set(system)
        
        true_positives = len(gold_set.intersection(system_set))
        false_positives = len(system_set - gold_set)
        false_negatives = len(gold_set - system_set)
        
        total_true_positives += true_positives
        total_false_positives += false_positives
        total_false_negatives += false_negatives
    
    precision = total_true_positives / (total_true_positives + total_false_positives) if (total_true_positives + total_false_positives) > 0 else 0
    recall = total_true_positives / (total_true_positives + total_false_negatives) if (total_true_positives + total_false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1, total_true_positives, total_false_positives, total_false_negatives


def type_f1_score(gold_standard, system_output):
    """
    Evaluates performance using Type Precision, Type Recall, and Type F1 score
    based on the set of unique terms extracted at least once across the entire dataset.
    """
    all_gold_terms = set()
    for item_terms in gold_standard:
        all_gold_terms.update(item_terms)
    
    all_system_terms = set()
    for item_terms in system_output:
        all_system_terms.update(item_terms)
    
    type_true_positives = len(all_gold_terms.intersection(all_system_terms))
    type_false_positives = len(all_system_terms - all_gold_terms)
    type_false_negatives = len(all_gold_terms - all_system_terms)
    
    type_precision = type_true_positives / (type_true_positives + type_false_positives) if (type_true_positives + type_false_positives) > 0 else 0
    type_recall = type_true_positives / (type_true_positives + type_false_negatives) if (type_true_positives + type_false_negatives) > 0 else 0
    type_f1 = 2 * (type_precision * type_recall) / (type_precision + type_recall) if (type_precision + type_recall) > 0 else 0
    
    return type_precision, type_recall, type_f1

## Evaluate Performance

In [5]:
# Prepare gold standard and predictions for evaluation
gold_standard = []
system_output = []

test_data = json.load(open(os.path.join(data_path, "taskA/test.json"), "r", encoding="utf-8"))
preds = json.load(open("predictions/subtaskA_a_vanilla_preds.json", "r", encoding="utf-8"))

for key in sorted(test_data.keys()):
    gold_standard.append(test_data[key]["terms"])
    system_output.append(preds[key]["terms"])

# Calculate metrics
precision, recall, f1, tp, fp, fn = micro_f1_score(gold_standard, system_output)
type_precision, type_recall, type_f1 = type_f1_score(gold_standard, system_output)

print("="*60)
print("VANILLA BASELINE RESULTS")
print("="*60)
print("\nMicro-averaged Metrics:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  TP={tp}, FP={fp}, FN={fn}")

print("\nType-level Metrics:")
print(f"  Type Precision: {type_precision:.4f}")
print(f"  Type Recall:    {type_recall:.4f}")
print(f"  Type F1 Score:  {type_f1:.4f}")
print("="*60)

VANILLA BASELINE RESULTS

Micro-averaged Metrics:
  Precision: 0.2946
  Recall:    0.4801
  F1 Score:  0.3651
  TP=9471, FP=22677, FN=10257

Type-level Metrics:
  Type Precision: 0.6122
  Type Recall:    0.1802
  Type F1 Score:  0.2784


## Example Predictions

In [6]:
# Show a few example predictions
print("Example Predictions:\n")
idx = 0
for id, content in preds.items():
    print(f"Document ID: {id}")
    print(f"Text: {content['text']}")
    print(f"Gold terms: {test_data[id]['terms'][:5]}") # Show first 5
    print(f"Predicted terms: {content['terms'][:5]}") # Show first 5
    
    correct = set(test_data[id]['terms']) & set(content['terms'])
    missed = set(test_data[id]['terms']) - set(content['terms'])
    wrong = set(content['terms']) - set(test_data[id]['terms'])

    print(f"Correct: {len(correct)}")
    print(f"Missed: {len(missed)}")
    print(f"Wrong: {len(wrong)}")
    print("-"*80, "\n")
    idx += 1
    if idx >= 5:  # Show only first 5 examples
        break

Example Predictions:

Document ID: mh_2
Text: Gut microbiota disturbances in hospitalized older adults with malnutrition and clinical outcomes.  OBJECTIVE: Malnutrition is one of the most threatening conditions in geriatric  populations. The gut microbiota has an important role in the host's metabolic and muscular health: however, its interplay with disease-related malnutrition is not well understood. We aimed to identify the association of malnutrition with the gut microbiota and predict clinical outcomes in hospitalized acutely ill older adults. METHODS: We performed a secondary longitudinal analysis in 108 geriatric patients from a prospective cohort evaluated at admission and 72 h of hospitalization. We collected clinical, demographic, nutritional, and 16S rRNA gene-sequenced gut microbiota data. Microbiota diversity, overall composition, and differential abundance were calculated and compared between patients with and without malnutrition. Microbiota features associated with malnu

In [8]:
from stop_words import get_stop_words
import json

preds = json.load(open("predictions/subtaskA_a_vanilla_preds.json", "r", encoding="utf-8"))
parsed_preds = {}
for id, content in preds.items():
    parsed_preds[id] = {
        "text": content['text'],
        "terms": []
    }
    for term in content['terms']:
        if term not in get_stop_words('en') and len(term) > 2 and term != ", and":
            parsed_preds[id]['terms'].append(term)

with open("predictions/subtaskA_a_vanilla_preds_filtered.json", "w", encoding="utf-8") as f:
    json.dump(parsed_preds, f, ensure_ascii=False, indent=4)

In [9]:
# Prepare gold standard and predictions for evaluation
gold_standard = []
system_output = []

test_data = json.load(open(os.path.join(data_path, "taskA/test.json"), "r", encoding="utf-8"))
preds = json.load(open("predictions/subtaskA_a_vanilla_preds_filtered.json", "r", encoding="utf-8"))

for key in sorted(test_data.keys()):
    gold_standard.append(test_data[key]["terms"])
    system_output.append(preds[key]["terms"])

# Calculate metrics
precision, recall, f1, tp, fp, fn = micro_f1_score(gold_standard, system_output)
type_precision, type_recall, type_f1 = type_f1_score(gold_standard, system_output)

print("="*60)
print("VANILLA BASELINE RESULTS")
print("="*60)
print("\nMicro-averaged Metrics:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  TP={tp}, FP={fp}, FN={fn}")

print("\nType-level Metrics:")
print(f"  Type Precision: {type_precision:.4f}")
print(f"  Type Recall:    {type_recall:.4f}")
print(f"  Type F1 Score:  {type_f1:.4f}")
print("="*60)

VANILLA BASELINE RESULTS

Micro-averaged Metrics:
  Precision: 0.3258
  Recall:    0.4786
  F1 Score:  0.3877
  TP=9442, FP=19540, FN=10286

Type-level Metrics:
  Type Precision: 0.6248
  Type Recall:    0.1785
  Type F1 Score:  0.2777
